# Day 3: from measured gates to a race plan

Use this notebook alongside [R1-R7](../../docs/race_practical.md), within the existing practical slots. Work from the team's measured gates, compare two planned paths, then analyse the recording of an actual attempt.

Use **Crazyflie lectures (.venv)**. These cells only read files, construct references and plot them: they never connect to a drone, arm it or fly. Run all works offline with an explicitly synthetic course. For hardware, substitute checked survey files and measured endpoints, then use the separate flight-tool checks in R5.

Before each edit, predict its effect. Choose a new `run_id` on a repeat and save this notebook with the results.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "code/day3/plan.py").is_file())
sys.path.insert(0, str(root / "code"))

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from trajectory_tools.course import construct_course, read_json
from day3.notebook_tools import plan_course, checked_plan
from trajectory_tools.preview import plot_trajectory

run_id = "trial-01"  # Change before repeating the notebook.
output = root / "day3/notebook" / run_id
output.mkdir(parents=True, exist_ok=False)
print("Python:", sys.executable)
print("Results:", output)


## R1: agree the reference frame, limits and roles

Open the [course and flight-command guide](../../docs/course_interface.md). Identify the Lighthouse origin/axes, resting place, planned airborne start/finish, drone size, physical obstacles, shared speed/acceleration limits, pilot and emergency-stop operator.

The file below is a **synthetic two-direction shuttle**, useful for learning the format. Replace it with the team's checked template for real course planning. Never use its positions as measured room coordinates or increase safety limits just to make a preview pass.


In [ ]:
baseline_plan, candidate_plan = None, None
template, course = None, None
template_path = root / "examples/course.shuttle.synthetic.json"
template = read_json(template_path)
display({key: template[key] for key in ("course_name", "start", "finish", "enabled_for_flight")})
display(template["limits"])


## R2: load measured gates and choose crossing order

Use the supplied [gate survey tool](../../docs/hoop_survey_challenge.md). Place the drone down, clear of propellers, confirm `STOP AND SURVEY`, and wait for **STOP VERIFIED** before handling it. The tool engages the firmware stop and leaves it engaged. Place the saved JSON paths below. Read the printed names, centres, normals and opening dimensions; cross-check them with manual measurements or an independent pass. Place the drone down and power-cycle before returning to normal flight preflight checks.

For measured inputs, list crossings as `name:+` or `name:-`, matching each survey's `name`. The sign sets the direction through that plane. Repeating the sequence makes laps. Empty lists below retain the template's **synthetic** crossings for offline work. Record any pointer offset and apply the documented correction to a copy, retaining the original survey.


In [ ]:
baseline_plan, candidate_plan = None, None
course = None  # Failed input edits invalidate downstream plans too.
course_provenance = None
survey_paths = []  # Example: [root / "survey/hoop-1.json"]
crossing_order = []  # Example: ["hoop-1:+", "hoop-1:-"]
laps = 2

surveys = [read_json(path) for path in survey_paths]
for survey in surveys:
    display({key: survey[key] for key in ("name", "centre", "normal", "radius_m",
                                          "shape", "width_m", "height_m") if key in survey})
print("Geometry:", "selected survey files; independent checks still needed" if surveys else "TEMPLATE ONLY; no measured survey selected")

course = construct_course(template, surveys, crossing_order, laps)
course_provenance = {
    "source_template": str(template_path),
    "source_surveys": [str(path) for path in survey_paths],
    "crossing_order": list(crossing_order),
    "laps": laps,
}
print("Ordered crossings:", [gate["name"] for gate in course["hoops"]])
print("Enabled for flight:", course["enabled_for_flight"])


## R3: inspect the conservative path

Predict how the approach distance and crossing normals shape the turns. The preview shows the openings, finite frame/bypass margins, start/finish, speed and acceleration. A gate's plane extends mathematically, but the obstacle check uses the gate's finite extent plus the stated margins.

The duration is a **geometric reference duration**, excluding takeoff and landing. The preview does not simulate the controller, positioning failures or race-time spread. A visually clear path still needs the physical checks in R5.


In [ ]:
baseline_plan, candidate_plan = None, None  # Rebuild the comparison after baseline edits.
baseline_plan = plan_course(course, provenance=course_provenance)
baseline_course, baseline_trajectory = checked_plan(baseline_plan)
figure = plot_trajectory(baseline_course, baseline_trajectory)
figure.savefig(output / "baseline.png", dpi=140)
display(figure)
plt.close(figure)
print("Planned duration [s]:", baseline_trajectory.duration_s)
print("Conservative speed bound [m/s]:", baseline_trajectory.speed_bound_m_s)
print("Conservative acceleration bound [m/s2]:", baseline_trajectory.acceleration_bound_m_s2)


## Change one planning choice

Choose a candidate target speed within the shared maximum, or change the approach distance in a separate comparison. Predict the duration and turn shape before running. Does the planner's acceleration-limited time scaling produce the improvement you expected?

The candidate starts from a copy of the validated baseline, preserving its geometry, approach distance, drone size, clearance and safety limits. This cell changes only the target speed. The planner is C2-continuous: jerk may jump at joins. Improving jerk continuity is an [optional extension](../../docs/trajectory_construction.md#stretch-goal-continuous-and-limited-jerk), not an additional core task.


In [ ]:
candidate_plan = None  # A failed rerun must not leave an older exportable plan.
candidate, candidate_trajectory = None, None
if baseline_plan is None:
    raise ValueError("Correct the course inputs and rerun baseline planning before comparing a candidate.")
candidate, baseline_trajectory = checked_plan(baseline_plan)  # Independent copy of the validated course.
candidate_target_speed = 0.4  # m/s; choose within the agreed maximum.
candidate["limits"]["target_speed_m_s"] = candidate_target_speed
candidate_plan = plan_course(candidate, provenance=baseline_plan.provenance)
candidate, candidate_trajectory = checked_plan(candidate_plan)

figure = plot_trajectory(candidate, candidate_trajectory)
figure.savefig(output / "candidate.png", dpi=140)
display(figure)
plt.close(figure)
print("Baseline / candidate reference duration [s]:",
      baseline_trajectory.duration_s, candidate_trajectory.duration_s)


## R4: compare and save the planned motion

Choose which candidate to keep, and explain the tradeoff using the plots and Day 1/2 results. The cell saves a disabled course, sampled reference and input choices. Each exported gate crossing keeps its own name and direction.

Select `candidate_plan` below, or `baseline_plan` to keep the validated baseline. A failed planning attempt invalidates that plan; correct its inputs and rerun planning before export. Edits to a validated plan also require replanning.

These files support the flight-tool dry run; that tool can generate a different path. Compare its own output before any hardware attempt. Planning success does not enable flight.


In [ ]:
selected_plan = candidate_plan  # Or baseline_plan.
selected_course, selected_trajectory = checked_plan(selected_plan)
times, positions, velocities, accelerations = selected_trajectory.sample(
    selected_course["limits"]["setpoint_rate_hz"])
assert selected_course["enabled_for_flight"] is False
(output / "course.json").write_text(json.dumps(selected_course, indent=2) + "\n")
np.savetxt(output / "trajectory.csv",
           np.column_stack((times, positions, velocities, accelerations)),
           delimiter=",", comments="",
           header="time_s,x_m,y_m,z_m,vx_m_s,vy_m_s,vz_m_s,ax_m_s2,ay_m_s2,az_m_s2")
summary = {
    "planned_duration_s": selected_trajectory.duration_s,
    "speed_bound_m_s": selected_trajectory.speed_bound_m_s,
    "acceleration_bound_m_s2": selected_trajectory.acceleration_bound_m_s2,
    "crossings": [{"name": gate["name"], "time_s": time}
                  for gate, time in zip(selected_course["hoops"], selected_trajectory.crossing_times_s)],
    **selected_plan.provenance,
    "note": "Planned reference only; no flight commands or physical qualification."
}
(output / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print("Saved disabled plan:", output / "course.json")


## R5-R6: leave the notebook for hardware checks and flight

Follow the [team flight checks and configuration deadlines](../../docs/race_practical.md#r5-qualify-and-freeze-the-race-configuration). Check the real supports/floor, full drone envelope, Lighthouse visibility, battery, measured mass/controller assumptions and actual tracking. Run the separate flight tool's dry run, complete the low-speed test and agree roles before enabling the checked configuration.

The notebook keeps `enabled_for_flight=false`. Use the established flight procedure, not a new notebook control loop. Retain failed/aborted attempts and the exact flown course/configuration. If checks or time run out, use the tested conservative setup or analyse a shared recording instead.


## R7: compare sent commands with the recorded trajectory

After an attempt, set both paths below to the telemetry CSV and the **exact course JSON used for that flight**. A shared checked recording works too. Do not substitute today's newly constructed candidate for a different flown course.

The existing offline plotter draws recorded setpoints, onboard position estimates and gates. Their difference includes timing and estimation effects; use separate crossing observations and the agreed race timing to judge physical success. This optional cell skips itself until both files are selected.


In [ ]:
telemetry_csv = None  # Example: root / "data/race-telemetry.csv"
flown_course_json = None  # The course actually used for that attempt.

if (telemetry_csv is None) != (flown_course_json is None):
    raise ValueError("Select both the telemetry and the exact flown course, or neither.")

if telemetry_csv is not None and flown_course_json is not None:
    prefix = output / "flown"
    subprocess.run([sys.executable, str(root / "scripts/plot_race.py"),
                    str(telemetry_csv), "--course", str(flown_course_json),
                    "--phase", "race", "--output", str(prefix), "--no-show"], check=True)
    display(Image(filename=str(output / "flown-trajectory.png")))
    display(Image(filename=str(output / "flown-errors.png")))
else:
    print("No flight recording selected. Keep the planned path labelled as a reference, not a flown result.")


## Discuss and keep

Save the edited notebook, checked survey files, course and plots. In the same short team notes, record the strategy, result, one failure or uncertainty and what you would change. Distinguish the reference duration, recorded tracking and independently observed crossings. Use [R7](../../docs/race_practical.md#r7-compare-planned-estimated-and-physical-performance) for the shared comparison; no extra report is needed.
